# NB_ingest_to_hubs — Silver → Hubs

Reads each Silver source table defined in the DV model config,
computes the SHA-256 hash key from business key columns, and performs
an INSERT-only MERGE into the corresponding vault hub table.
No updates — hubs are insert-only by DV 2.0 design.

In [ ]:
%run ../helpers/NB_catalog_helpers

In [ ]:
%run ./NB_dv_metadata

In [ ]:
dbutils.widgets.text("CATALOG", "workspace", "Unity Catalog name")
dbutils.widgets.text("VAULT_SCHEMA", "vault", "Vault schema")
dbutils.widgets.text("SILVER_SCHEMA", "silver", "Silver schema")
dbutils.widgets.text("MODEL_PATH", "pipeline_configs/datavault/dv_model.json", "DV Model JSON Path")
dbutils.widgets.text("WATERMARK_TS", "", "Optional: only process records >= this timestamp")

CATALOG       = dbutils.widgets.get("CATALOG")
VAULT_SCHEMA  = dbutils.widgets.get("VAULT_SCHEMA")
SILVER_SCHEMA = dbutils.widgets.get("SILVER_SCHEMA")
MODEL_PATH    = dbutils.widgets.get("MODEL_PATH")
WATERMARK_TS  = dbutils.widgets.get("WATERMARK_TS")

In [ ]:
model = load_model(MODEL_PATH)
print(f"Loaded {len(model['hubs'])} hubs from model")

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

hub_counts = {}

for hub_cfg in model['hubs']:
    if not hub_cfg.get('enabled', True):
        print(f"  Skipping disabled hub: {hub_cfg['name']}")
        continue

    hub_name    = hub_cfg['name']
    source_tbl  = hub_cfg['source_table']
    bk_cols     = hub_cfg['business_key_columns']
    load_dt_col = hub_cfg['load_date_column']
    rec_src     = hub_cfg['record_source']
    hk_alias    = f"HK_{hub_name[4:]}"
    target_tbl  = f"{CATALOG}.{VAULT_SCHEMA}.{hub_name.lower()}"

    # Ensure target table exists
    create_hub_table(hub_cfg)

    # Read Silver source
    src_df = spark.table(source_tbl)
    if WATERMARK_TS:
        src_df = src_df.filter(F.col(load_dt_col) >= WATERMARK_TS)

    # Compute hash key
    src_df = (
        src_df
        .withColumn(hk_alias, generate_hash_key(bk_cols))
        .withColumn('LOAD_DATE', F.col(load_dt_col).cast('timestamp'))
        .withColumn('RECORD_SOURCE', F.lit(rec_src))
        .select(hk_alias, *bk_cols, 'LOAD_DATE', 'RECORD_SOURCE')
        .dropDuplicates([hk_alias])
    )

    # INSERT-only MERGE (DV 2.0: hubs never update)
    hub_tbl = DeltaTable.forName(spark, target_tbl)
    (
        hub_tbl.alias('tgt')
        .merge(src_df.alias('src'), f"tgt.{hk_alias} = src.{hk_alias}")
        .whenNotMatchedInsertAll()
        .execute()
    )

    count = spark.table(target_tbl).count()
    hub_counts[hub_name] = count
    print(f"  {hub_name}: {count:,} total rows in {target_tbl}")

In [ ]:
print('\nHub ingestion complete.')
for hub_name, cnt in hub_counts.items():
    print(f'  {hub_name}: {cnt:,} rows')